# Práctica 6 - Inteligencia Artificial

## Tema 5: Introducción al aprendizaje automático

### Implementación de un clasificador Naive Bayes

José Luis Ruiz Reina

En este ejercicio implementaremos como construir un clasificador Naive Bayes suponiendo que todos los atributos que describen los datos son discretos (toman valor entre un conjunto finito de opciones). En particular lo usaremos para construir un clasificador que prediga el partido al que pertenece un congresista estadounidense (demócrata o republicano), a partir de sus votaciones. 

La descripción del entrenamiento y predicción del modelo Naive Bayes está en el tema 4. 

#### Conjuntos de datos

Para describir un conjunto de entranamiento $D = \{d_1, \dots, d_n\}$ usaremos dos listas `X_D` e `y_D`. Un elemento, $D_i$, del conjunto de entranamiento estará representado por la pareja de elementos: `X_D[i-1]` e `y_D[i-1]`. 
Si consideramos que los atributos $\{A_1, \dots, A_m, C\}$ describen los datos (distinguimos el último como valor de clasificación) un elemento del conjunto de entranamiento $d = (a_1, \dots, a_m, c)$ (para ese dato el valor de cada atributo $A_j$ es $a_j$ y su valor de clasificación es $c$), el correspondiente elemento de `X_D` será la lista `[a1, ..., am]` y el de `y_D` será `c`.

Importamos un par de conjuntos de datos. En primer lugar, los que usan en la explicación del tema 4 (día adecuado para jugar al tenis).


| Ej. | Cielo | Temperatura | Humedad | Viento | JugarTenis |
| :--- | :--- | :--- | :--- | :--- | :---: |
| $d_1$ | Soleado | Alta | Alta | Débil | - |
| $d_2$ | Soleado | Alta | Alta | Fuerte | - |
| $d_3$ | Nublado | Alta | Alta | Débil | + |
| $d_4$ | Lluvia | Suave | Alta | Débil | + |
| $d_5$ | Lluvia | Fresca | Normal | Débil | + |
| $d_6$ | Lluvia | Fresca | Normal | Fuerte | - |
| $d_7$ | Nublado | Fresca | Normal | Fuerte | + |
| $d_8$ | Soleado | Suave | Alta | Débil | - |
| $d_9$ | Soleado | Fresca | Normal | Débil | + |
| $d_{10}$ | Lluvia | Suave | Normal | Débil | + |
| $d_{11}$ | Soleado | Suave | Normal | Fuerte | + |
| $d_{12}$ | Nublado | Suave | Alta | Fuerte | + |
| $d_{13}$ | Nublado | Alta | Normal | Débil | + |
| $d_{14}$ | Lluvia | Suave | Alta | Fuerte | - |

$d_{14}$ vendría dado por `X_tenis[13] == ['Lluvia', 'Suave', 'Alta', 'Fuerte']` e `y_tenis[13] = '-'`


In [4]:
from jugar_tenis import X_tenis, y_tenis
X_tenis[13], y_tenis[13]

(['Lluvia', 'Suave', 'Alta', 'Fuerte'], 'no')

Vamos a usar también para probar la implementación, un conocido conjunto de datos. Se trata de un conjunto con las 16 votaciones (sobre distintos temas) realizadas a lo largo del año 1984 por 435 congresistas en Estados Unidos, junto con el partido al que pertenecen (republicano o demócrata) 

Los valores posibles de cada voto son *sí* (`'s'`), no (`'n'`) y presente (`'?'`). El voto *presente*, en el congreso de Estados Unidos, podríamos asimilarlo al voto en blanco (aunque no es exactamente lo mismo).   

In [5]:
from votos import X_votos, y_votos
# El primero de los ejemplos:
X_votos[0], y_votos[0]

(['n',
  's',
  'n',
  's',
  's',
  's',
  'n',
  'n',
  'n',
  's',
  '?',
  's',
  's',
  's',
  'n',
  's'],
 'republicano')

#### Ejercicio 1

Definir una función `prob_prior(c, y_D)` que estime la probabilidad de un determinado valor de clasificación, `c`, a partir de las clasificaciones de un conjunto de entrenamiento, recogidas en la lista `y_D`.
$$P(C = c) = \frac{n(D_c)}{n(D)}$$
siendo $D$ el conjunto de entrenamiento, $n$ el tamaño de un conjunto y $D_c = \{d \in D |$ el valor de clasificación de $d$ es $c \}$

In [6]:
# -------   Solución:
def prob_prior(c, y_D):
    res = 0
    for y in y_D:
        if y==c:
            res = res + 1

    return res/len(y_D)

# Ejemplos:
prob_prior("si", y_tenis)
# 0.6428571428571429

0.6428571428571429

In [7]:
prob_prior("demócrata", y_votos)
# 0.6137931034482759

0.6137931034482759

#### Ejercicio 2

Definir una función `prob_cond(X_D, y_D, Ai, a, c)` que devuelve la estimación de la probabilidad de que un atributo tome un valor, condicionado a un determinado valor de clasificación. 

$$P(A = a | C = c) = \frac{n(D_a \cap D_c)}{D_c}$$

Por ejemplo: $P($ Cielo = Soleado | JugarTenis = + $) = \frac{2}{9} = 0'2$ y $P($ Viento = Fuerte | JugarTenis = - $) = \frac{3}{5} = 0'6$ 

La función recibe:
- `X_D` y `y_D`: los datos del conjunto de entrenamiento $D$ según lo descrito anteriormente.
- `Ai`: el índice que ocupa el valor del atributo $A$ en cada elemento de `X_D`. Por ejemplo, en el conjunto de datos de jugar al tenis, índice para los valores del atributo 'Cielo' es el 0, y el del atributo Viento es el 3. 
- `a`: el valor del atributo para el que se está estimando la probabilidad.
- `c`: el valor de clasificación que condiciona la probabilidad que se está estimando.


In [8]:
# ----  Solución:

def prob_cond(X_D, y_D, Ai, a, c, k=0):     # k=0 K LO PONEMOS A 0 SIEMPRE EN ESTE LAB
    n_c = 0
    n_ac = 0
    for x, y in zip(X_D, y_D):
        if y == c:
            n_c += 1
            if x[Ai] == a:
                n_ac += 1
    return n_ac / n_c

                # PROFESOR
    # matrib_X_Ai = len(set([x[Ai] for x in X_D]))
    # D_c = sum(y==c for y in y_D) + k*matrib_X_Ai
    # return sum(c==y for x,y in zip(X_D, y_D) if x[Ai] == a)/D_c


# Ejemplos:
# * Probabilidad de Cielo soleado, dado que es un día bueno para jugar al tenis
prob_cond(X_tenis, y_tenis, 0, "Soleado", "si")
# 0.2222222222222222

0.2222222222222222

In [9]:
# * Probabilidad de Viento fuerte, dado que no es un día bueno para jugar al tenis
prob_cond(X_tenis, y_tenis, 3, "Fuerte", "no")
# 0.6

0.6

#### Ejercicio 3

Para calcular todas las probabilidades a priori y condicionadas, podríamos llamar a las funciones anteriores para todas las combinaciones posibles de atributos, valores y clasificaciones. Pero si el conjunto de datos es muy grande, esto sería ineficiente, ya que cada llamada independiente realiza un recorrido por todo el conjunto de datos.

Sería aconsejable calcular todas las probabilidades de una única pasada. Se pide definir una función `calcula_probs(X_D, y_D)` que devuelva todas las probabilidades estimadas, a partir del conjunto de datos representado por las listas `X_D` e `y_D`, recorriendo los datos una única vez. 

Devolver el resultado en forma de dos diccionarios:

- Un diccionario cuyas claves son las clasificaciones y los valores son las probabilidades a priori de cada una de ellas.
- Un diccionario cuyas claves son las ternas `(Ai, a, c)` y cuyos valores son las correspondientes probabilidades condicionadas. 

Algunos elementos de python que pueden ser de utilidad:
- `set`: permite crear un conjunto a partir de un iterador. Para más información consultar [Set Types](https://docs.python.org/3/builtins/stdtypes.html#set-types-set-frozenset)
- `zip`: permite iterar en paralelo sobre varios iteradores, proporcionando tuplas con un elmento de cada uno. Para más información consultar [zip](https://docs.python.org/3/builtins/functions.html#zip)
- `*`: permite 'desempaquetar' los elementos de un iterador para pasarlos como argumentos independientes a una función. Para más información consultar [Unpacking in Lists and List Comprehensions](https://docs.python.org/3/tutorial/datastructures.html#unpacking-in-lists-and-list-comprehensions)
- `enumerate`: a partir de un iterador devuelve otro emparejando cada elemento del primero con su posición. Para más información consultar [enumerate](https://docs.python.org/3/builtins/functions.html#enumerate)

A continuación se incluyen algunos ejemplos de uso:

In [10]:
lista = [[1,'a',0], [2, 'b', 0], [3, 'c', 0]]
set([1,2,3,4,2,3,4,2,3,1,2,3,2,4,1]), list(zip(*lista)), list(enumerate(lista))

({1, 2, 3, 4},
 [(1, 2, 3), ('a', 'b', 'c'), (0, 0, 0)],
 [(0, [1, 'a', 0]), (1, [2, 'b', 0]), (2, [3, 'c', 0])])

In [11]:
# ------ Solución:


def calcula_probs(X_D, y_D):
    prior_d = {}
    prob_cond_d = {}
    
    # 1. Obtenemos las clases únicas (por ejemplo: 'si', 'no' o 'demócrata', 'republicano')
    clases = set(y_D)
    
    # 2. Calculamos las probabilidades a priori iterando de forma clásica
    for c in clases:
        prior_d[c] = prob_prior(c, y_D)
        
    # 3. Calculamos las probabilidades condicionadas
    num_atributos = len(X_D[0])
    
    # Bucle externo: recorre cada índice de atributo (0 para Cielo, 1 para Temperatura...)
    for Ai in range(num_atributos):
        
        # Bucle para extraer los valores únicos que toma este atributo en concreto
        valores_atributo = set()
        for x in X_D:
            valores_atributo.add(x[Ai])
            
        # Bucles anidados: para cada valor único y para cada clase, calculamos prob_cond
        for a in valores_atributo:
            for c in clases:
                # Guardamos el resultado en el diccionario usando la tupla como clave
                prob_cond_d[(Ai, a, c)] = prob_cond(X_D, y_D, Ai, a, c, k=0)
                
    return prior_d, prob_cond_d


            # PROFESOR
# def calcula_probs(X_D, y_D):
#     prior_d = {c:prob_prior(c,y_D) for c in set(y_D)}
#     prob_cond_d = {(Ai,a,c): prob_cond(X_D, y_D, Ai, a, c, k=0)
#                  for Ai in range(len(X_D[0]))
#                  for a in set([x[Ai] for x in X_D])
#                  for c in set(y_D)}
#     return prior_d, prob_cond_d

calcula_probs(X_tenis, y_tenis)
# ({'+': 0.6428571428571429, '-': 0.35714285714285715},
#  {(0, 'Soleado', '+'): 0.2222222222222222,
#   (0, 'Soleado', '-'): 0.6,
#   (0, 'Nublado', '+'): 0.4444444444444444,
#   (0, 'Nublado', '-'): 0.0,
#   (0, 'Lluvia', '+'): 0.3333333333333333,
#   (0, 'Lluvia', '-'): 0.4,
#   (1, 'Alta', '+'): 0.2222222222222222,
#   (1, 'Alta', '-'): 0.4,
#   (1, 'Baja', '+'): 0.3333333333333333,
#   (1, 'Baja', '-'): 0.2,
#   (1, 'Suave', '+'): 0.4444444444444444,
#   (1, 'Suave', '-'): 0.4,
#   (2, 'Alta', '+'): 0.3333333333333333,
#   (2, 'Alta', '-'): 0.8,
#   (2, 'Normal', '+'): 0.6666666666666666,
#   (2, 'Normal', '-'): 0.2,
#   (3, 'Fuerte', '+'): 0.3333333333333333,
#   (3, 'Fuerte', '-'): 0.6,
#   (3, 'Débil', '+'): 0.6666666666666666,
#   (3, 'Débil', '-'): 0.4})

({'si': 0.6428571428571429, 'no': 0.35714285714285715},
 {(0, 'Soleado', 'si'): 0.2222222222222222,
  (0, 'Soleado', 'no'): 0.6,
  (0, 'Lluvia', 'si'): 0.3333333333333333,
  (0, 'Lluvia', 'no'): 0.4,
  (0, 'Nublado', 'si'): 0.4444444444444444,
  (0, 'Nublado', 'no'): 0.0,
  (1, 'Suave', 'si'): 0.4444444444444444,
  (1, 'Suave', 'no'): 0.4,
  (1, 'Alta', 'si'): 0.2222222222222222,
  (1, 'Alta', 'no'): 0.4,
  (1, 'Baja', 'si'): 0.3333333333333333,
  (1, 'Baja', 'no'): 0.2,
  (2, 'Normal', 'si'): 0.6666666666666666,
  (2, 'Normal', 'no'): 0.2,
  (2, 'Alta', 'si'): 0.3333333333333333,
  (2, 'Alta', 'no'): 0.8,
  (3, 'Fuerte', 'si'): 0.3333333333333333,
  (3, 'Fuerte', 'no'): 0.6,
  (3, 'Débil', 'si'): 0.6666666666666666,
  (3, 'Débil', 'no'): 0.4})

#### Ejercicio 4

Una vez que hemos estimado las probabilidades a partir de los datos, vamos a definir una función `predicción_NB(probprior,probcond,ejemplo)` que a partir de las probabilidades calculadas, devuelve la clasificación que el modelo Naive Bayes predice para un ejemplo dado $(a_1, \dots, a_n, c)$.

$$c_{NB}= \mathop{argmax}_{c\in dom(C)} P(C = c)\prod_{i=1\dots n} P(A_i = a_i| C = c)$$


In [13]:
# --------- Solución:

def predicción_NB(probprior, probcond, ejemplo):
    mejor_prob = -1
    mejor_clase = None
    
    # Probamos con cada clase posible (por ejemplo: '+', '-')
    for c in probprior.keys():
        # Empezamos con la probabilidad a priori de esa clase
        prob_actual = probprior[c]
        
        # Multiplicamos por la probabilidad condicionada de cada atributo del ejemplo
        for Ai in range(len(ejemplo)):
            a = ejemplo[Ai]
            # Usamos get() por si un valor no existe en el diccionario (devolvería 0)
            prob_actual = prob_actual * probcond.get((Ai, a, c), 0)
            
        # Si esta clase obtiene mejor probabilidad que la anterior, la guardamos
        if prob_actual > mejor_prob:
            mejor_prob = prob_actual
            mejor_clase = c
            
    return mejor_clase

probpriors_tenis, probconds_tenis = calcula_probs(X_tenis, y_tenis)
predicción_NB(probpriors_tenis, probconds_tenis, ["Soleado", "Suave", "Alta", "Fuerte"])
# '-'

'no'

#### Ejercicio 5

Recopilando lo realizado en los ejercicios anteriores, definir una clase `NaiveBayes` con dos métodos:

- Un método `entrena` para la estimación de las probabilidades
- Un método `predicción` para predecir la clasificación de nuevos ejemplos

Estimar las probabilidades usando suavizado. La constante `k` de suavizado debe ser un argumento del constructor (por defecto `k=1`). 


In [14]:
# ----- Solución
        
class NaiveBayes():
    def __init__(self, k=1):
        self.k = k
        self.probprior = {}
        self.probcond = {}
        self.clases = set()

    def entrena(self, X_D, y_D):
        self.clases = set(y_D)
        
        # 1. Calcular probabilidades a priori
        for c in self.clases:
            casos_c = 0
            for y in y_D:
                if y == c:
                    casos_c += 1
            self.probprior[c] = casos_c / len(y_D)
            
        # 2. Calcular probabilidades condicionadas con Suavizado
        num_atributos = len(X_D[0])
        for Ai in range(num_atributos):
            
            # Sacamos los valores posibles de este atributo para saber el tamaño |Ai|
            valores_atributo = set()
            for x in X_D:
                valores_atributo.add(x[Ai])
            V = len(valores_atributo)  # Esto es |Ai| en la fórmula de Laplace
            
            for c in self.clases:
                # Contamos cuántos ejemplos totales hay de esta clase
                n_c = 0
                for y in y_D:
                    if y == c:
                        n_c += 1
                        
                for a in valores_atributo:
                    # Contamos cuántos coinciden en atributo y clase a la vez
                    n_ac = 0
                    for x, y in zip(X_D, y_D):
                        if y == c and x[Ai] == a:
                            n_ac += 1
                    
                    # Aplicamos la fórmula con el suavizado aditivo +k
                    self.probcond[(Ai, a, c)] = (n_ac + self.k) / (n_c + self.k * V)
    
    def predicción(self, ejemplo):
        mejor_prob = -1
        mejor_clase = None
        
        for c in self.clases:
            prob_actual = self.probprior[c]
            for Ai in range(len(ejemplo)):
                a = ejemplo[Ai]
                prob_actual = prob_actual * self.probcond.get((Ai, a, c), 0)
                
            if prob_actual > mejor_prob:
                mejor_prob = prob_actual
                mejor_clase = c
                
        return mejor_clase
    
nb_tenis = NaiveBayes()
nb_tenis.entrena(X_tenis, y_tenis)
nb_tenis.predicción(["Soleado","Suave","Alta","Fuerte"])
# '-'

'no'

#### Ejercicio 6

Vamos a aplicar ahora el clasificador Naive Bayes al conjunto de datos de los votos. Pero antes, vamos a separar (aleatoriamente) una parte de los datos para entrenamiento, dejando otra parte para evaluar el rendimiento del clasificador obtenido. 

Para ello, definir una función `divide_entrenamiento_prueba(X_D, y_D, prop_test)`, que devuelve `X_entr`, `y_entr`, `X_test`, `y_test`, resultado de dividir `X_D` e `y_D` en dos partes, aleatoriamente. El argumento `prop_test` marca la proporción del total de datos que se apartan para test (por defecto $0'2$).  

**Nota**: usar `random.shuffle`

In [15]:
# ---------   Solución:

import random

def divide_entrenamiento_prueba(X_D, y_D, prop_test=0.2):
    # Juntamos cada dato con su etiqueta para barajarlos sin que se desordenen entre ellos
    datos = list(zip(X_D, y_D))
    random.shuffle(datos)
    
    # Calculamos en qué índice cortar (por ejemplo, el 80% de los datos)
    corte = int(len(datos) * (1 - prop_test))
    
    X_entr = []
    y_entr = []
    # Rellenamos las listas de entrenamiento desde el principio hasta el corte
    for i in range(corte):
        X_entr.append(datos[i][0])
        y_entr.append(datos[i][1])
        
    X_test = []
    y_test = []
    # Rellenamos las listas de test desde el corte hasta el final
    for i in range(corte, len(datos)):
        X_test.append(datos[i][0])
        y_test.append(datos[i][1])
        
    return X_entr, y_entr, X_test, y_test

X_votos_entr, y_votos_entr, X_votos_test, y_votos_test = divide_entrenamiento_prueba(X_votos,y_votos)
len(X_votos), len(X_votos_entr), len(X_votos_test)
# (435, 348, 87)

(435, 348, 87)

#### Ejercicio 7

Entrenar un clasificador Naive Bayes sobre el conjunto de entrenamiento de los votos, y probar algunas predicciones sobre ejemplos del conjunto de prueba, comparando la predicción con el valor de clasificación que se conoce.


In [16]:
# ---- Ejemplo (pueden ser otros resultados, debido a la aleatoriedad):


# Suponiendo que tienes cargados X_votos e y_votos del archivo de tu profe
X_votos_entr, y_votos_entr, X_votos_test, y_votos_test = divide_entrenamiento_prueba(X_votos, y_votos, 0.2)

# Creamos el clasificador con suavizado k=1
nb_votos = NaiveBayes(k=1)
nb_votos.entrena(X_votos_entr, y_votos_entr)

# Imprimimos de 10 en 10 algunos ejemplos para ver qué hace
for i in range(0, min(90, len(X_votos_test)), 10):
    pred = nb_votos.predicción(X_votos_test[i])
    real = y_votos_test[i]
    print(f"Ejemplo {i}. Predicción: {pred}. Clase real: {real}.")
    


# Ejemplo 0. Predicción: demócrata. clase real: demócrata.
# Ejemplo 10. Predicción: demócrata. clase real: demócrata.
# Ejemplo 20. Predicción: republicano. clase real: republicano.
# Ejemplo 30. Predicción: republicano. clase real: demócrata.
# Ejemplo 40. Predicción: demócrata. clase real: demócrata.
# Ejemplo 50. Predicción: demócrata. clase real: demócrata.
# Ejemplo 60. Predicción: republicano. clase real: republicano.
# Ejemplo 70. Predicción: demócrata. clase real: demócrata.
# Ejemplo 80. Predicción: republicano. clase real: republicano.

Ejemplo 0. Predicción: demócrata. Clase real: demócrata.
Ejemplo 10. Predicción: republicano. Clase real: republicano.
Ejemplo 20. Predicción: republicano. Clase real: demócrata.
Ejemplo 30. Predicción: demócrata. Clase real: demócrata.
Ejemplo 40. Predicción: demócrata. Clase real: demócrata.
Ejemplo 50. Predicción: republicano. Clase real: republicano.
Ejemplo 60. Predicción: demócrata. Clase real: demócrata.
Ejemplo 70. Predicción: demócrata. Clase real: demócrata.
Ejemplo 80. Predicción: demócrata. Clase real: demócrata.


#### Ejercicio 8

Definir una función `rendimiento(clasificador, X_T, y_T)`, que devuelve la proporción de aciertos de un clasificador (como los que define la clase `NaiveBayes`), sobre un conjunto de datos $T$. 

Calcular el rendimiento del clasificador Naive Bayes aprendido para el conjunto de los votos.

In [19]:
# ----- Solución:

def rendimiento(clasificador, X_T, y_T):
    aciertos = 0
    
    # Repasamos todo el conjunto de prueba
    for x, y_real in zip(X_T, y_T):
        # Le pedimos al modelo que adivine
        y_pred = clasificador.predicción(x)
        
        # Si acierta, sumamos uno
        if y_pred == y_real:
            aciertos += 1
            
    # Devolvemos el porcentaje de acierto
    return aciertos / len(X_T)

# Puede salir otro resultado, debido a la aleatoriedad
rendimiento(nb_votos, X_votos_test, y_votos_test)
# 0.8735632183908046

0.9080459770114943

In [20]:
media = 0
for _ in range(1000):
    X_votos_entr, y_votos_entr, X_votos_test, y_votos_test = divide_entrenamiento_prueba(X_votos, y_votos)
    nb_votos.entrena(X_votos_entr, y_votos_entr)
    media += rendimiento(nb_votos, X_votos_test, y_votos_test)
media / 1000

# Aprox. 0'89977

0.9005172413793096